# T10 It3: targeted verification, how much could it win? (headroom before any GPU work)

Idea: spend extra matching effort only when **two distant candidates are both plausible** (the choice and the best candidate > 4 sigma away
both have judge P >= 0.3 or >= 0.5), e.g. a second LoFTR look at both (T7 `second_look.py`). Before building it: how often does that happen,
and how many points could a perfect tie-breaker win? Judge trained on NUDT design sets only (out-of-fold on them); official TRY checked; PC
(M = 10) and Jetson (M = 5, context from the 5 matched candidates).

In [1]:
import os, sys
os.environ.setdefault("OMP_NUM_THREADS", "8")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250)
import train.checks.candidate_judge as cj
from train.checks.holdout import NUDT_DESIGN, load_t8_table
RES = ROOT / "tests/Test_Iteration_10/results"
T = load_t8_table(ROOT, locked=True)

In [2]:
def pairs(t, p, m=None):
    t = t.assign(p=p)
    if m: t = t[t["rank"] < m]
    rows = []
    for (d, ph), g in t.groupby(["data", "photo"]):
        g = g[g.inliers >= 3]
        if not len(g): continue
        c = g.iloc[int(np.lexsort((-g.coarse.to_numpy(), -(g.inliers + g.z).to_numpy()))[0])]
        far = g[np.hypot(g.x - c.x, g.y - c.y) > 4 * c.sigma]
        r = far.loc[far.p.idxmax()] if len(far) else None
        rows.append(dict(data=d, photo=ph, p_choice=c.p, err_choice=c.err_sigma, p_rival=r.p if r is not None else 0.0, err_rival=r.err_sigma if r is not None else np.nan))
    return pd.DataFrame(rows)
p10, _ = cj.judge_probability(T, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
T5 = cj.add_context(T, m=5); p5, _ = cj.judge_probability(T5, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
rows = []
for var, (t, p, m, tau) in {"pc": (T, p10, None, 0.9), "jetson": (T5, p5, 5, 0.9)}.items():
    o = pairs(t, p, m)
    for thr in (0.3, 0.5):
        trig = o[(o.p_choice >= thr) & (o.p_rival >= thr)]
        for d, g in trig.groupby("data"):
            undecided = g[g.p_choice < tau]
            rows.append(dict(var=var, threshold=thr, data=d, photos=o[o.data == d].shape[0], fires=len(g), already_answered=int((g.p_choice >= tau).sum()),
                             choice_right=int((g.err_choice <= 4).sum()), rival_right=int((g.err_rival <= 4).sum()),
                             undecided=len(undecided), undecided_rival_right=int((undecided.err_rival <= 4).sum()),
                             ceiling_points_per_60=round(6 * 10 * len(undecided) / o[o.data == d].shape[0], 2)))
H = pd.DataFrame(rows); H.to_csv(RES / "it3_targeted_verification_headroom.csv", index=False); H

,var,threshold,data,photos,fires,already_answered,choice_right,rival_right,undecided,undecided_rival_right,ceiling_points_per_60
0,pc,0.3,nudt101,297,34,33,34,0,1,0,0.2
1,pc,0.3,nudt202,200,23,22,23,0,1,0,0.3
2,pc,0.3,nudt404,200,18,17,17,1,1,1,0.3
3,pc,0.3,nudt505,300,20,19,19,1,1,1,0.2
4,pc,0.3,nudt606,300,28,27,27,0,1,0,0.2
5,pc,0.3,official_try,181,6,6,6,0,0,0,0.0
6,pc,0.5,nudt101,297,24,24,24,0,0,0,0.0
7,pc,0.5,nudt202,200,16,16,16,0,0,0,0.0
8,pc,0.5,nudt404,200,11,11,11,0,0,0,0.0
9,pc,0.5,nudt505,300,10,9,9,1,1,1,0.2


**Read:** the trigger mostly fires on photos the judge already answers with P ~ 1 (the right place is strong and a look-alike is also
plausible); the choice is right there. Photos it would actually decide (choice below the cut) are ~1 per NUDT set and 0 on official TRY, so a
perfect tie-breaker could add at most ~0.2 points per 60 per set. Not built: the GPU cost is not justified by the ceiling.